# TA-003B — ResNet-50 Reentrenado con GAN Augmentation

**Objetivo:** Reentrenar ResNet-50 con `train_augmented.csv` y comparar AUC contra baseline sin augmentation (AUC test = 0.8619).

**Novedades vs TA-003:** Dataset aumentado (6,206 imgs), VetXRayDataset maneja DICOM + PNG, AMP, checkpoints cada 10 épocas, W&B.

In [1]:
import json
import time
import numpy as np
import pandas as pd
import pydicom
import wandb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from pathlib import Path
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from tqdm import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

BASE          = Path(r'E:\Taller Integrador I\ModelosIA\Radiografias')
CHECKPOINTS   = Path(r'E:\Taller Integrador I\ModelosIA\modelos\checkpoints')
NOTEBOOKS_DIR = Path(r'E:\Taller Integrador I\ModelosIA\modelos\Notebooks')
TRAIN_DIR     = BASE / 'dataset_split' / 'train'
VAL_DIR       = BASE / 'dataset_split' / 'val'
TEST_DIR      = BASE / 'dataset_split' / 'test'
MANIFESTS     = BASE / 'dataset_split' / 'manifests'

CLASSES               = ['alveolar_pattern', 'bronchial_pattern', 'pleural_effusion', 'cardiomegaly', 'no_finding']
NUM_CLASSES           = len(CLASSES)
SEED                  = 42
BATCH_SIZE            = 32
LR                    = 1e-4
WEIGHT_DECAY          = 1e-4
EPOCHS                = 30
PATIENCE              = 10
FEATURE_EXTRACTION_EPOCHS = 5
DROPOUT               = 0.4
CHECKPOINT_EVERY      = 10
RESUME_FROM           = None

torch.manual_seed(SEED)
np.random.seed(SEED)
print('Config OK')

Device: cuda
GPU: NVIDIA GeForce RTX 4060
Config OK


In [2]:
wandb.init(
    project='vetxray-cnn',
    entity='dbaylont1-antenor-orrego-private-university',
    name='TA-003B-ResNet50-Augmented',
    config={
        'task': 'TA-003B', 'model': 'ResNet-50',
        'dataset': 'VetXRay-GAN-Augmented',
        'batch_size': BATCH_SIZE, 'lr': LR,
        'weight_decay': WEIGHT_DECAY, 'epochs': EPOCHS,
        'patience': PATIENCE, 'fe_epochs': FEATURE_EXTRACTION_EPOCHS,
        'dropout': DROPOUT, 'amp': True,
        'augmentation': 'GAN', 'baseline_auc': 0.8619, 'seed': SEED
    },
    tags=['ResNet50', 'augmented', 'TA-003B', 'VetXRay']
)
print('W&B run:', wandb.run.url)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Pcuser\_netrc.
wandb: Currently logged in as: dbaylont1 (dbaylont1-antenor-orrego-private-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B run: https://wandb.ai/dbaylont1-antenor-orrego-private-university/vetxray-cnn/runs/sljrjhdz


In [3]:
TRANSFORM_TRAIN = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
TRANSFORM_VAL = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class VetXRayDataset(Dataset):
    def __init__(self, df, transform, train_dir=TRAIN_DIR):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.train_dir = train_dir
        self.labels    = self._build_labels()

    def _build_labels(self):
        labels = np.zeros((len(self.df), NUM_CLASSES), dtype=np.float32)
        for i, row in self.df.iterrows():
            for j, cls in enumerate(CLASSES):
                if isinstance(row['TAG'], str) and cls in row['TAG']:
                    labels[i, j] = 1.0
        return labels

    def _load_dicom(self, path):
        dcm = pydicom.dcmread(str(path))
        arr = dcm.pixel_array.astype(np.float32)
        if hasattr(dcm, 'PhotometricInterpretation') and dcm.PhotometricInterpretation == 'MONOCHROME1':
            arr = arr.max() - arr
        p2, p98 = np.percentile(arr, 2), np.percentile(arr, 98)
        arr = np.clip(arr, p2, p98)
        arr = (arr - p2) / (p98 - p2 + 1e-8)
        img = Image.fromarray((arr * 255).astype(np.uint8))
        img = img.resize((224, 224), Image.LANCZOS)
        return Image.merge('RGB', [img, img, img])

    def _load_png(self, path):
        img = Image.open(path).convert('L')
        img = img.resize((224, 224), Image.LANCZOS)
        return Image.merge('RGB', [img, img, img])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        is_syn = row.get('is_synthetic', False)
        syn_path = row.get('synthetic_path', None)
        if is_syn and pd.notna(syn_path):
            img_rgb = self._load_png(syn_path)
        else:
            img_rgb = self._load_dicom(self.train_dir / row['FileName'])
        return self.transform(img_rgb), torch.tensor(self.labels[idx], dtype=torch.float32)

print('VetXRayDataset OK')

VetXRayDataset OK


In [4]:
df_train = pd.read_csv(MANIFESTS / 'train_augmented.csv')
df_val   = pd.read_csv(MANIFESTS / 'val.csv')
df_test  = pd.read_csv(MANIFESTS / 'test.csv')

print(f'Train augmented: {len(df_train)}')
print(f'Val:             {len(df_val)}')
print(f'Test:            {len(df_test)}')

pos_counts = np.array([df_train['TAG'].str.contains(c, na=False).sum() for c in CLASSES], dtype=np.float32)
neg_counts = len(df_train) - pos_counts
class_weights = torch.tensor(neg_counts / (pos_counts + 1e-8), dtype=torch.float32).to(DEVICE)
print('\nClass weights:')
for c, w in zip(CLASSES, class_weights.cpu()): print(f'  {c}: {w:.3f}')

train_dataset = VetXRayDataset(df_train, TRANSFORM_TRAIN, train_dir=TRAIN_DIR)
val_dataset   = VetXRayDataset(df_val,   TRANSFORM_VAL,   train_dir=VAL_DIR)
test_dataset  = VetXRayDataset(df_test,  TRANSFORM_VAL,   train_dir=TEST_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)
print('DataLoaders OK')

Train augmented: 6206
Val:             1063
Test:            940

Class weights:
  alveolar_pattern: 5.206
  bronchial_pattern: 5.206
  pleural_effusion: 7.866
  cardiomegaly: 4.541
  no_finding: 1.514
DataLoaders OK


In [5]:
def build_resnet50():
    m = models.resnet50(weights='IMAGENET1K_V1')
    for p in m.parameters(): p.requires_grad = False
    m.fc = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(m.fc.in_features, NUM_CLASSES))
    return m.to(DEVICE)

model = build_resnet50()
total = sum(p.numel() for p in model.parameters())
train_ = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Params totales:     {total:,}')
print(f'Params entrenables: {train_:,} (feature extraction)')

Params totales:     23,518,277
Params entrenables: 10,245 (feature extraction)


In [6]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, all_probs, all_labels = 0.0, [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            with torch.amp.autocast('cuda'):
                out  = model(imgs)
                loss = criterion(out, labels)
            total_loss += loss.item() * imgs.size(0)
            all_probs.append(torch.sigmoid(out).cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    probs  = np.vstack(all_probs)
    labels = np.vstack(all_labels)
    avg_loss = total_loss / len(loader.dataset)
    try:    auc = roc_auc_score(labels, probs, average='macro')
    except: auc = 0.0
    preds = (probs >= 0.5).astype(int)
    f1  = f1_score(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels.flatten(), preds.flatten())
    return avg_loss, auc, f1, acc, probs, labels


def save_ckpt(epoch, model, opt, sched, scaler, best_auc, history, tag):
    p = CHECKPOINTS / f'resnet50_aug_{tag}_ep{epoch}.pth'
    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                'optimizer_state_dict': opt.state_dict(),
                'scheduler_state_dict': sched.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
                'best_auc': best_auc, 'history': history}, p)
    return p


def load_ckpt(path, model, opt, sched, scaler):
    c = torch.load(path, map_location=DEVICE)
    model.load_state_dict(c['model_state_dict'])
    opt.load_state_dict(c['optimizer_state_dict'])
    sched.load_state_dict(c['scheduler_state_dict'])
    scaler.load_state_dict(c['scaler_state_dict'])
    print(f'Retomando desde epoch {c["epoch"]} | best_auc {c["best_auc"]:.4f}')
    return c['epoch'], c['best_auc'], c['history']


print('Helpers OK')

Helpers OK


In [7]:
def train_loop(model, optimizer, scheduler, scaler, criterion,
               start_ep, end_ep, best_auc, history, phase):
    patience_counter = 0
    for epoch in range(start_ep, end_ep):
        model.train()
        train_loss = 0.0
        t0 = time.time()
        for imgs, labels in tqdm(train_loader, desc=f'Ep {epoch+1}/{end_ep}', leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                out  = model(imgs)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() * imgs.size(0)

        train_loss /= len(train_loader.dataset)
        val_loss, val_auc, val_f1, val_acc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step(val_auc)
        elapsed = (time.time() - t0) / 60
        lr = optimizer.param_groups[0]['lr']

        print(f'[{phase}] Ep {epoch+1:3d}/{end_ep} | '
              f'Train {train_loss:.4f} | Val {val_loss:.4f} | '
              f'AUC {val_auc:.4f} | F1 {val_f1:.4f} | LR {lr:.2e} | {elapsed:.1f}min')

        entry = {'epoch': epoch+1, 'phase': phase, 'train_loss': train_loss,
                 'val_loss': val_loss, 'val_auc': val_auc, 'val_f1': val_f1, 'val_acc': val_acc}
        history.append(entry)
        wandb.log({'epoch': epoch+1, 'phase': phase, 'train_loss': train_loss,
                   'val_loss': val_loss, 'val_auc': val_auc, 'val_f1': val_f1,
                   'val_acc': val_acc, 'lr': lr})

        if val_auc > best_auc:
            best_auc = val_auc
            patience_counter = 0
            save_ckpt(epoch+1, model, optimizer, scheduler, scaler, best_auc, history, 'best')
            print(f'  >>> Mejor AUC: {best_auc:.4f} — guardado')
        else:
            patience_counter += 1

        if (epoch + 1) % CHECKPOINT_EVERY == 0:
            save_ckpt(epoch+1, model, optimizer, scheduler, scaler, best_auc, history, 'periodic')

        if patience_counter >= PATIENCE:
            print(f'Early stopping en epoch {epoch+1}')
            break

    return best_auc, history


print('train_loop() OK')

train_loop() OK


In [8]:
criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                               lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
scaler    = torch.amp.GradScaler('cuda')

start_epoch, best_auc, history = 0, 0.0, []

if RESUME_FROM and Path(RESUME_FROM).exists():
    start_epoch, best_auc, history = load_ckpt(RESUME_FROM, model, optimizer, scheduler, scaler)

print(f'=== FASE 1: Feature Extraction ({FEATURE_EXTRACTION_EPOCHS} epocas) ===')
best_auc, history = train_loop(model, optimizer, scheduler, scaler, criterion,
                                start_epoch, FEATURE_EXTRACTION_EPOCHS,
                                best_auc, history, 'feature_extraction')

=== FASE 1: Feature Extraction (5 epocas) ===


[feature_extraction] Ep   1/5 | Train 1.0544 | Val 0.9485 | AUC 0.6087 | F1 0.2337 | LR 1.00e-04 | 15.7min
  >>> Mejor AUC: 0.6087 — guardado


[feature_extraction] Ep   2/5 | Train 0.9767 | Val 0.9467 | AUC 0.6351 | F1 0.2531 | LR 1.00e-04 | 15.8min
  >>> Mejor AUC: 0.6351 — guardado


[feature_extraction] Ep   3/5 | Train 0.9452 | Val 0.9329 | AUC 0.6459 | F1 0.2637 | LR 1.00e-04 | 14.8min
  >>> Mejor AUC: 0.6459 — guardado


[feature_extraction] Ep   4/5 | Train 0.9231 | Val 0.9323 | AUC 0.6593 | F1 0.2752 | LR 1.00e-04 | 14.1min
  >>> Mejor AUC: 0.6593 — guardado


[feature_extraction] Ep   5/5 | Train 0.9157 | Val 0.9301 | AUC 0.6670 | F1 0.2701 | LR 1.00e-04 | 14.7min
  >>> Mejor AUC: 0.6670 — guardado


In [9]:
EPOCHS = 30

In [10]:
print('=== FASE 2: Fine-tuning completo ===')
for p in model.parameters(): p.requires_grad = True
optimizer = torch.optim.AdamW(model.parameters(), lr=LR/5, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
print(f'Params entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

best_auc, history = train_loop(model, optimizer, scheduler, scaler, criterion,
                                FEATURE_EXTRACTION_EPOCHS, EPOCHS,
                                best_auc, history, 'fine_tuning')
print(f'Entrenamiento completo. Mejor AUC val: {best_auc:.4f}')

=== FASE 2: Fine-tuning completo ===
Params entrenables: 23,518,277


[fine_tuning] Ep   6/30 | Train 0.7296 | Val 0.7359 | AUC 0.8141 | F1 0.4661 | LR 2.00e-05 | 14.8min
  >>> Mejor AUC: 0.8141 — guardado


[fine_tuning] Ep   7/30 | Train 0.5567 | Val 0.6868 | AUC 0.8416 | F1 0.5175 | LR 2.00e-05 | 14.8min
  >>> Mejor AUC: 0.8416 — guardado


[fine_tuning] Ep   8/30 | Train 0.4781 | Val 0.6431 | AUC 0.8714 | F1 0.5435 | LR 2.00e-05 | 14.8min
  >>> Mejor AUC: 0.8714 — guardado


[fine_tuning] Ep   9/30 | Train 0.4302 | Val 0.6418 | AUC 0.8643 | F1 0.5767 | LR 2.00e-05 | 14.8min


...

[fine_tuning] Ep  17/30 | Train 0.1746 | Val 0.8135 | AUC 0.8697 | F1 0.5905 | LR 1.00e-05 | 15.6min


[fine_tuning] Ep  18/30 | Train 0.1652 | Val 0.8266 | AUC 0.8664 | F1 0.5969 | LR 1.00e-05 | 15.7min


[fine_tuning] Ep  19/30 | Train 0.1454 | Val 0.8341 | AUC 0.8622 | F1 0.5958 | LR 5.00e-06 | 15.8min


[fine_tuning] Ep  20/30 | Train 0.1284 | Val 0.8483 | AUC 0.8667 | F1 0.5888 | LR 5.00e-06 | 15.6min


[fine_tuning] Ep  21/30 | Train 0.1286 | Val 0.8849 | AUC 0.8649 | F1 0.5876 | LR 5.00e-06 | 15.5min
Early stopping en epoch 21
Entrenamiento completo. Mejor AUC val: 0.8747


In [11]:
best_files = sorted(CHECKPOINTS.glob('resnet50_aug_best*.pth'))
if best_files:
    ckpt = torch.load(best_files[-1], map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Mejor modelo cargado: {best_files[-1].name}')

t0 = time.time()
test_loss, test_auc, test_f1, test_acc, test_probs, test_labels = evaluate(model, test_loader, criterion)
inf_ms = (time.time() - t0) / len(test_dataset) * 1000

print(f'\n=== RESULTADOS TEST ===')
print(f'AUC macro:  {test_auc:.4f}  (baseline: 0.8619)')
print(f'F1 macro:   {test_f1:.4f}')
print(f'Accuracy:   {test_acc:.4f}')
print(f'Inferencia: {inf_ms:.2f} ms/imagen')

try:
    aucs = roc_auc_score(test_labels, test_probs, average=None)
    print('\nAUC por clase:')
    for c, a in zip(CLASSES, aucs): print(f'  {c}: {a:.4f}')
except Exception as e:
    print(e)

wandb.log({'test_auc': test_auc, 'test_f1': test_f1, 'test_accuracy': test_acc,
           'inference_ms': inf_ms, 'mejora_vs_baseline': test_auc - 0.8619})

Mejor modelo cargado: resnet50_aug_best_ep8.pth

=== RESULTADOS TEST ===
AUC macro:  0.8831  (baseline: 0.8619)
F1 macro:   0.5797
Accuracy:   0.8428
Inferencia: 137.73 ms/imagen

AUC por clase:
  alveolar_pattern: 0.8732
  bronchial_pattern: 0.7938
  pleural_effusion: 0.9788
  cardiomegaly: 0.8904
  no_finding: 0.8792


In [12]:
results = {
    'model': 'resnet50', 'task': 'TA-003B',
    'dataset': 'VetXRay-GAN-Augmented',
    'baseline_auc': 0.8619,
    'val_auc': best_auc,
    'test_auc': test_auc,
    'test_f1_macro': test_f1,
    'test_accuracy': test_acc,
    'inference_ms': inf_ms,
    'mejora_auc': round(test_auc - 0.8619, 4),
    'history': history
}
out = NOTEBOOKS_DIR / 'resnet50_aug_results.json'
with open(out, 'w') as f: json.dump(results, f, indent=4)

mejora = test_auc - 0.8619
print(f'Resultados: {out}')
print(f'AUC test: {test_auc:.4f} vs baseline 0.8619 => {"MEJORA" if mejora > 0 else "SIN MEJORA"} de {abs(mejora):.4f}')

wandb.finish()
print('TA-003B completado.')

Resultados: E:\Taller Integrador I\ModelosIA\modelos\Notebooks\resnet50_aug_results.json
AUC test: 0.8831 vs baseline 0.8619 => MEJORA de 0.0212


epoch,▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇██
inference_ms,▁
lr,█████▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
mejora_vs_baseline,▁
test_accuracy,▁
test_auc,▁
test_f1,▁
train_loss,█▇▇▇▇▆▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
val_acc,▂▁▁▁▁▄▆▇▇▇▇█▇▇███████
val_auc,▁▂▂▂▃▆▇██████████████
+2,...


TA-003B completado.
